# Bin 1 vs bin 2 contamination-proxy diagnostics for `g-z`

This notebook compares the first two radial bins using contamination proxies that avoid splitting on foreground luminosity or halo-mass-correlated foreground properties:

- high vs low `deblend_fluxOverlapFraction` from DP1 Object metadata, retrieved through TAP on the Rubin Science Platform when available;
- high vs low background `g`-band brightness;
- high vs low background `z`-band brightness;
- red vs blue foreground, using foreground `g-z` color only.

For each proxy and each radial bin, the notebook computes a no-random/no-reference, flipped-corrected pair estimator for the high and low proxy groups. In practice this is `forward - flipped`, where the forward term stacks the background `g/z` flux ratios for the selected pairs and the flipped term stacks the foreground `g/z` flux ratios for those same pairs. The reported contrast is then `Delta = (forward - flipped)_high - (forward - flipped)_low`. For background-brightness splits, high background flux should dilute fractional leaked foreground light: high `g`-band background flux has the blue-leakage expected sign of positive `Delta`, while high `z`-band background flux has the red-leakage expected sign of negative `Delta`. The final section summarizes what the metrics say about contamination in bin 1 relative to bin 2.

The reported `delta/sigma_jk` value normalizes the high-minus-low proxy contrast by the regular bin jackknife uncertainty. The notebook also keeps `delta/E(g-z)` as a stress-test amplitude relative to the measured signal, not as a direct fractional bias in the full stack.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if (ROOT / "src").exists() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from dusty_colors.color_split_bias import (
    HIGH_GROUP,
    LOW_GROUP,
    analyze_proxy_split_bin,
    summarize_inner_bin_exclusion_rerun,
    summarize_inner_bin_pair_influence,
    summarize_pair_influence_table,
    attach_inner_bin_exclusion_flags,
    attach_proxy_signal_normalization,
    attach_independent_proxy_columns,
    build_radial_pair_table,
    load_saved_stack_bin_context,
    proxy_contamination_summary_table,
    proxy_definitions,
    radial_bin_label,
)
from dusty_colors.postage_stamps import (
    available_tap_columns,
    fetch_object_metadata,
    requested_blend_columns,
    select_existing_columns,
)

STACK_ID = "dp1_default"
COLOR = "g-z"
MODE = "fcolors"
RADIAL_BIN_INDICES = (0, 1)
OVERLAP_BANDS = ("r", "i")
BOOTSTRAP_SAMPLES = 5000
PERMUTATION_SAMPLES = 5000
RANDOM_SEED = 20260612

STACK_DIR = ROOT / "results" / "stacks" / STACK_ID
PAIR_DIR = ROOT / "results" / "postage_stamps" / STACK_ID
OUT_DIR = PAIR_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = STACK_DIR / "config_resolved.yaml"
OBJECT_METADATA_PATH = OUT_DIR / "object_blend_metadata.csv"

## Load stack inputs and build pair tables

In [ ]:
with open(CONFIG_PATH) as stream:
    resolved = yaml.safe_load(stream)

analysis = resolved["analysis"]["data"]
sample_id = resolved["sample"]["data"]["id"]
stack_config = analysis["stack"]
r_edges = np.asarray(stack_config["r_bin_edges"], dtype=float)
snr_max = float(stack_config.get("snr_max", 100.0))

sample_dir = ROOT / "results" / "samples" / sample_id
foreground = pd.read_parquet(sample_dir / "foreground.parquet")
background = pd.read_parquet(sample_dir / "background.parquet")

pair_tables = {}
pair_summary_rows = []
for bin_index in RADIAL_BIN_INDICES:
    r_min = float(r_edges[bin_index])
    r_max = float(r_edges[bin_index + 1])
    pairs = build_radial_pair_table(
        foreground,
        background,
        stack_id=STACK_ID,
        bin_index=bin_index,
        r_edges_kpc=r_edges,
    )
    pair_tables[bin_index] = pairs
    pair_summary_rows.append(
        {
            "bin_number": bin_index + 1,
            "bin_label": radial_bin_label(bin_index, r_min, r_max),
            "r_min_kpc": r_min,
            "r_max_kpc": r_max,
            "n_positional_pairs": len(pairs),
        }
    )

all_object_ids = pd.unique(
    pd.concat(
        [
            *(pairs["foreground_object_id"] for pairs in pair_tables.values()),
            *(pairs["background_object_id"] for pairs in pair_tables.values()),
        ]
    )
    .dropna()
    .astype("int64")
)

print(f"stack: {STACK_ID}")
print(f"sample: {sample_id}")
print(f"unique objects needing blend metadata: {len(all_object_ids)}")
display(pd.DataFrame(pair_summary_rows))
display(pair_tables[RADIAL_BIN_INDICES[0]].head())

## Retrieve or load DP1 Object blend metadata

This cell works on the Rubin Science Platform. If RSP TAP imports are unavailable, or if TAP retrieval fails, the notebook continues with local-only proxies and marks the overlap proxy as unavailable.

In [ ]:
object_metadata = pd.DataFrame()
metadata_status = "not_attempted"
missing_object_columns = []

if OBJECT_METADATA_PATH.exists():
    object_metadata = pd.read_csv(OBJECT_METADATA_PATH)
    metadata_status = f"loaded {OBJECT_METADATA_PATH}"
else:
    try:
        from lsst.rsp import get_tap_service

        tap_service = get_tap_service("tap")
        available_columns = available_tap_columns(tap_service)
        requested_columns = requested_blend_columns(OVERLAP_BANDS)
        object_columns = select_existing_columns(requested_columns, available_columns)
        missing_object_columns = sorted(set(requested_columns) - set(object_columns))
        object_metadata = fetch_object_metadata(
            tap_service, all_object_ids, object_columns
        )
        object_metadata.to_csv(OBJECT_METADATA_PATH, index=False)
        metadata_status = f"fetched {len(object_metadata)} Object rows from TAP"
    except Exception as exc:
        metadata_status = f"metadata unavailable: {exc}"

print(metadata_status)
if missing_object_columns:
    print("Missing requested Object columns:", missing_object_columns)
display(
    object_metadata.head()
    if not object_metadata.empty
    else pd.DataFrame({"status": [metadata_status]})
)

## Attach proxy inputs to each pair table

In [ ]:
pair_inputs = {}
proxy_input_rows = []
for bin_index, pairs in pair_tables.items():
    prepared = attach_independent_proxy_columns(
        pairs,
        foreground,
        background,
        color=COLOR,
        snr_max=snr_max,
        object_metadata=object_metadata,
        overlap_bands=OVERLAP_BANDS,
    )
    pair_inputs[bin_index] = prepared
    proxy_input_rows.append(
        {
            "bin_number": bin_index + 1,
            "n_pairs": len(prepared),
            "n_valid_background_gz": int(
                prepared["valid_background_stack_color"].sum()
            ),
            "n_finite_overlap": int(
                np.isfinite(prepared["max_deblend_fluxOverlapFraction"]).sum()
            ),
            "n_finite_background_g_flux": int(
                np.isfinite(prepared["log10_background_g_flux"]).sum()
            ),
            "n_finite_background_z_flux": int(
                np.isfinite(prepared["log10_background_z_flux"]).sum()
            ),
            "n_finite_foreground_gz": int(
                np.isfinite(prepared["foreground_g_z"]).sum()
            ),
        }
    )

display(pd.DataFrame(proxy_input_rows))
display(
    pair_inputs[RADIAL_BIN_INDICES[0]][
        [
            "pair_id",
            "theta_arcsec",
            "foreground_g_z",
            "background_g_z",
            "log10_background_g_flux",
            "log10_background_z_flux",
            "max_deblend_fluxOverlapFraction",
        ]
    ].head()
)

## Compute high-vs-low proxy stacks for bins 1 and 2

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
proxy_results = []
proxy_result_lookup = {}
for definition in proxy_definitions(COLOR):
    proxy = definition["proxy"]
    proxy_result_lookup[proxy] = {}
    for bin_index in RADIAL_BIN_INDICES:
        result = analyze_proxy_split_bin(
            pair_inputs[bin_index],
            stack_id=STACK_ID,
            bin_index=bin_index,
            r_min_kpc=float(r_edges[bin_index]),
            r_max_kpc=float(r_edges[bin_index + 1]),
            proxy=proxy,
            metric_col=definition["metric_col"],
            high_label=definition["high_label"],
            low_label=definition["low_label"],
            color=COLOR,
            bootstrap_samples=BOOTSTRAP_SAMPLES,
            permutation_samples=PERMUTATION_SAMPLES,
            rng=rng,
            expected_positive_delta=definition["expected_positive_delta"],
        )
        proxy_result_lookup[proxy][bin_index] = result
        summary = dict(result["summary"])
        summary["description"] = definition["description"]
        summary["metadata_required"] = definition["requires_metadata"]
        proxy_results.append(summary)

proxy_summary = pd.DataFrame(proxy_results)
saved_context = load_saved_stack_bin_context(STACK_DIR, color=COLOR, mode=MODE)
if not saved_context.empty:
    proxy_summary = proxy_summary.merge(
        saved_context,
        on=["bin_index", "bin_number"],
        how="left",
    )

proxy_summary = attach_proxy_signal_normalization(proxy_summary)

summary_cols = [
    "proxy",
    "estimator",
    "bin_number",
    "n_proxy_valid",
    "n_high",
    "n_low",
    "split_threshold",
    "high_stack_gz",
    "low_stack_gz",
    "delta_high_minus_low_gz_mag",
    "regular_corrected_gz_signal_mag",
    "regular_corrected_gz_signal_err_mag",
    "delta_over_regular_corrected_gz_err",
    "delta_over_regular_corrected_gz_signal",
    "bootstrap_z",
    "permutation_p_two_sided",
    "expected_contamination_sign_detected",
    "saved_forward_gz",
    "saved_corrected_gz",
]
display(proxy_summary[[col for col in summary_cols if col in proxy_summary.columns]])
proxy_summary.to_csv(
    OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_bin2_proxy_contamination_summary.csv",
    index=False,
)

## Per-proxy stack tables and pair labels

In [ ]:
prefix = COLOR.replace("-", "_")
for proxy, per_bin in proxy_result_lookup.items():
    for bin_index, result in per_bin.items():
        label = result["summary"]["bin_label"]
        result["stack_table"].to_csv(OUT_DIR / f"{prefix}_{label}_{proxy}_stack.csv")
        result["split_pairs"].to_csv(
            OUT_DIR / f"{prefix}_{label}_{proxy}_pairs.csv", index=False
        )
        print(f"{proxy}, bin {bin_index + 1}")
        display(result["stack_table"])
        display(
            result["split_pairs"][
                [
                    "pair_id",
                    "theta_arcsec",
                    "r_perp_kpc",
                    result["group_col"],
                    "foreground_g_z",
                    "background_g_z",
                    "log10_background_g_flux",
                    "log10_background_z_flux",
                    "max_deblend_fluxOverlapFraction",
                ]
            ].head(8)
        )

## Bin 1 vs bin 2 summary

In [ ]:
interpretation = proxy_contamination_summary_table(proxy_summary)
display(interpretation)
interpretation.to_csv(
    OUT_DIR
    / f"{COLOR.replace('-', '_')}_bin1_bin2_proxy_contamination_interpretation.csv",
    index=False,
)

for _, row in interpretation.iterrows():
    print(f"\n{row['proxy']}")
    print(row["interpretation"])

## Inner-bin exclusion rerun

This section recomputes the bin-1 one-bin forward stack after excluding potentially blended pairs. The approximate corrected signal keeps the original reference/random correction fixed and applies only the inner-bin forward-stack change, so it is an inner-bin-only robustness diagnostic rather than a full radial-profile rerun.


In [ ]:
INNER_BIN_INDEX = 0
REVIEW_LABELS_PATH = OUT_DIR / "review_labels.csv"

if REVIEW_LABELS_PATH.exists():
    review_labels = pd.read_csv(REVIEW_LABELS_PATH)
    review_label_status = f"loaded {REVIEW_LABELS_PATH}"
else:
    review_labels = pd.DataFrame()
    review_label_status = (
        "review_labels.csv not found; manual-label exclusions are inactive"
    )
print(review_label_status)

if saved_context.empty:
    inner_saved_context = {}
else:
    inner_context_rows = saved_context.loc[
        saved_context["bin_index"].eq(INNER_BIN_INDEX)
    ]
    inner_saved_context = (
        inner_context_rows.iloc[0].to_dict() if not inner_context_rows.empty else {}
    )

inner_exclusion_pairs = attach_inner_bin_exclusion_flags(
    pair_inputs[INNER_BIN_INDEX],
    review_labels=review_labels,
)
inner_exclusion_summary = summarize_inner_bin_exclusion_rerun(
    inner_exclusion_pairs,
    saved_context=inner_saved_context,
    color=COLOR,
)

exclusion_cols = [
    "scenario",
    "n_valid_original",
    "n_excluded_valid",
    "n_kept_valid",
    "excluded_valid_fraction",
    "full_forward_gz",
    "kept_forward_gz",
    "delta_kept_minus_full_forward_gz",
    "regular_corrected_gz_signal_mag",
    "regular_corrected_gz_signal_err_mag",
    "approx_corrected_gz_after_exclusion",
    "approx_delta_over_regular_corrected_gz_err",
    "approx_delta_over_regular_corrected_gz_signal",
    "approx_corrected_over_regular_corrected_gz_signal",
]
display(inner_exclusion_summary[exclusion_cols])

inner_exclusion_pairs.to_csv(
    OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_inner_exclusion_pairs.csv",
    index=False,
)
inner_exclusion_summary.to_csv(
    OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_inner_exclusion_rerun_summary.csv",
    index=False,
)

## Inner-bin influence check

This leave-one-out diagnostic removes each valid bin-1 pair one at a time, recomputes the one-bin forward stack, and reports the approximate corrected-signal shift in mag and in units of the regular bin jackknife uncertainty.


In [ ]:
inner_influence = summarize_inner_bin_pair_influence(
    inner_exclusion_pairs,
    saved_context=inner_saved_context,
    color=COLOR,
)
inner_influence_summary = summarize_pair_influence_table(inner_influence)

display(inner_influence_summary)

influence_cols = [
    "pair_id",
    "theta_arcsec",
    "r_perp_kpc",
    "delta_leave_one_out_minus_full_forward_gz",
    "approx_delta_over_regular_corrected_gz_err",
    "approx_delta_over_regular_corrected_gz_signal",
    "approx_corrected_gz_after_pair_removal",
    "max_deblend_fluxOverlapFraction",
    "log10_background_g_flux",
    "log10_background_z_flux",
    "foreground_g_z",
    "background_g_z",
    "exclude_overlap_top10pct",
    "exclude_overlap_top20pct",
    "exclude_same_parent",
    "exclude_deblend_failed",
]
display(
    inner_influence[
        [col for col in influence_cols if col in inner_influence.columns]
    ].head(12)
)

inner_influence.to_csv(
    OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_inner_influence_pairs.csv",
    index=False,
)
inner_influence_summary.to_csv(
    OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_inner_influence_summary.csv",
    index=False,
)

## Diagnostic plot

In [ ]:
plot_data = proxy_summary.copy()
plot_data["bin_label_plot"] = "bin " + plot_data["bin_number"].astype(str)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), constrained_layout=True)
for ax, proxy in zip(axes, plot_data["proxy"].drop_duplicates()):
    sub = plot_data.loc[plot_data["proxy"].eq(proxy)].sort_values("bin_number")
    x = np.arange(len(sub))
    colors = [
        "#b5524a" if flag else "#4f83b7"
        for flag in sub["expected_contamination_sign_detected"]
    ]
    ax.bar(x, sub["delta_high_minus_low_gz_mag"], color=colors, alpha=0.85)
    ax.axhline(0.0, color="black", linewidth=1.0)
    ax.set_xticks(x, sub["bin_label_plot"])
    ax.set_ylabel("high minus low stacked bg g-z [mag]")
    ax.set_title(proxy)
    for xi, (_, row) in zip(x, sub.iterrows()):
        ax.text(
            xi,
            row["delta_high_minus_low_gz_mag"],
            f"p={row['permutation_p_two_sided']:.2g}",
            ha="center",
            va="bottom" if row["delta_high_minus_low_gz_mag"] >= 0 else "top",
            fontsize=9,
        )

plot_path = OUT_DIR / f"{COLOR.replace('-', '_')}_bin1_bin2_proxy_contamination.png"
fig.savefig(plot_path, dpi=160)
plot_path